# P1 — TRAIN

이 notebook은 운영진 배포 데이터에서 scratch 학습하는 과정을 독립적으로 보존합니다. 최종 답안을 복사하는 notebook이 아닙니다. 기본 실행은 이미 생성된 certified model manifest와 입력 해시를 검증하며, `RUN_FULL_SCRATCH_RETRAIN=True`로 바꾸면 별도 출력 폴더에 새 모델을 학습합니다. Certified 모델은 운영진 배포 train의 776,706행, 165개 past-only feature로 학습된 3-seed MS-TCN입니다.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

PACKAGE_DIR = Path.cwd().resolve().parent
DATA_DIR = PACKAGE_DIR / '01_data' / 'organizer_dataset'
if not (PACKAGE_DIR / 'contract.json').is_file():
    raise RuntimeError('Run this notebook from P?/02_train')
sys.path.insert(0, str(Path.cwd().resolve()))
contract = json.loads((PACKAGE_DIR / 'contract.json').read_text(encoding='utf-8'))
input_manifest = json.loads((PACKAGE_DIR / '01_data/INPUT_MANIFEST.json').read_text(encoding='utf-8'))
model_manifest = json.loads((PACKAGE_DIR / '03_model/MODEL_MANIFEST.json').read_text(encoding='utf-8'))
print({'candidate': contract['candidate_id'], 'input_files': len(input_manifest['files']), 'model_status': model_manifest['status']})

## 1. 현재 학습 산출물의 무결성 확인

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

checked = []
for record in contract['model_files']:
    path = PACKAGE_DIR / record['path']
    assert path.is_file() and sha256(path) == record['sha256']
    checked.append(record['path'])
print({'verified_model_files': len(checked), 'pretrained_weights_loaded': 0, 'external_rows': 0})

## 2. 선택적 전체 scratch 재학습

기존 certified weights를 덮어쓰지 않도록 별도 디렉터리에 출력합니다. P1은 3×150 epoch라 오래 걸립니다. P3의 간단 wrapper는 base branches를 재학습하며, 최종 router/calibrator의 정확한 역사적 학습 소스는 `07_source/scripts/`에 함께 있습니다.

In [ ]:
RUN_FULL_SCRATCH_RETRAIN = False
OUTPUT_DIR = PACKAGE_DIR / '03_model/retrained_from_scratch'
if RUN_FULL_SCRATCH_RETRAIN:
    import train_model
    retrain_receipt = train_model.train(DATA_DIR, PACKAGE_DIR, OUTPUT_DIR)
    print(retrain_receipt)
else:
    print({'status': 'CERTIFIED_TRAINING_OUTPUT_VERIFIED', 'retrain_executed_now': False, 'toggle': 'set RUN_FULL_SCRATCH_RETRAIN=True'})

## 3. 다음 단계

최종 제출 재현은 `../04_predict/PREDICT.ipynb`가 `03_model`의 검증된 가중치를 실제로 로드해 수행합니다. `05_answer`는 모델 추론 SHA가 최고점 후보 SHA와 같을 때만 생성됩니다.